In [3]:
import itertools
import json
import re
import shutil
import subprocess
import tempfile
from fractions import Fraction
from pathlib import Path

import sympy as sp

import pytest

from pyope import OPE, Zero, simplify, simplify_with_wolfram


from pyope import (
    BasicOperator,
    Bosonic,
    Fermionic,
    MakeOPE,
    NO,
    OPE,
    One,
    Zero,
    bracket,
    check_jacobi_identity,
    clear_registry,
    d,
    simplify,
)

In [8]:
W_Z3_OPE_GROUND_TRUTH = {
    "T_T": {"4": "-15/2 * One", "2": "2 * T", "1": "Derivative[1][T]"},
    "T_J": {"2": "J", "1": "Derivative[1][J]"},
    "J_J": {"2": "-5 * One"},
    "T_G": {"2": "3/2 * G", "1": "Derivative[1][G]"},
    "T_Gbar": {"2": "3/2 * Gbar", "1": "Derivative[1][Gbar]"},
    "T_W": {"2": "3/2 * W", "1": "Derivative[1][W]"},
    "T_Wbar": {"2": "3/2 * Wbar", "1": "Derivative[1][Wbar]"},
    "T_GW": {"2": "2 * GW", "1": "Derivative[1][GW]"},
    "T_GbarWbar": {"2": "2 * GbarWbar", "1": "Derivative[1][GbarWbar]"},
    "J_G": {"1": "-G"},
    "J_Gbar": {"1": "Gbar"},
    "J_W": {"1": "3 * W"},
    "J_Wbar": {"1": "-3 * Wbar"},
    "J_GW": {"1": "2 * GW"},
    "J_GbarWbar": {"1": "-2 * GbarWbar"},
    "W_Wbar": {
        "3": "-20/9 * One",
        "2": "4/3 * J",
        "1": "2/3 * T - 1/3 * NO[J, J] + 2/3 * Derivative[1][J]",
    },
    "W_G": {"1": "-GW"},
    "Wbar_Gbar": {"1": "-GbarWbar"},
    "G_W": {"1": "GW"},
    "Gbar_W": {},
    "Gbar_Wbar": {"1": "GbarWbar"},
    "G_Wbar": {},
    "G_Gbar": {
        "3": "5 * One",
        "2": "J",
        "1": "-T + 1/2 * Derivative[1][J]",
    },
    "Gbar_GW": {"2": "-3 * W", "1": "-Derivative[1][W]"},
    "G_GbarWbar": {"2": "-3 * Wbar", "1": "-Derivative[1][Wbar]"},
    "Wbar_GW": {
        "2": "4/3 * G",
        "1": "2/3 * NO[J, G] + 2/3 * Derivative[1][G]",
    },
    "W_GbarWbar": {
        "2": "-4/3 * Gbar",
        "1": "2/3 * NO[J, Gbar] - 2/3 * Derivative[1][Gbar]",
    },
    "GW_GbarWbar": {
        "4": "-20/3 * One",
        "3": "8/3 * J",
        "2": "2 * T - 1/3 * NO[J, J] + 4/3 * Derivative[1][J]",
        "1": "2/3 * NO[G, Gbar] - 2/3 * NO[J, T] - 1/3 * NO[J, Derivative[1][J]] + 4/3 * Derivative[1][T] + 1/3 * Derivative[2][J]",
    },
}

In [4]:


class _WLParser:
    def __init__(self, text, symbols):
        self.tokens = re.findall(
            r"Derivative|NO|One|[A-Za-z][A-Za-z0-9]*|\d+/\d+|\d+|\[|\]|\(|\)|,|\+|-|\*",
            text,
        )
        self.pos = 0
        self.symbols = symbols

    def peek(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def pop(self, expected=None):
        token = self.peek()
        if expected is not None and token != expected:
            raise ValueError(f"Expected {expected!r}, got {token!r}")
        self.pos += 1
        return token

    def parse(self):
        expr = self.parse_expr()
        if self.peek() is not None:
            raise ValueError(f"Unexpected trailing token {self.peek()!r}")
        return expr

    def parse_expr(self):
        value = self.parse_term()
        while self.peek() in {"+", "-"}:
            op = self.pop()
            right = self.parse_term()
            value = value + right if op == "+" else value - right
        return value

    def parse_term(self):
        value = self.parse_factor()
        while self.peek() == "*":
            self.pop("*")
            value = value * self.parse_factor()
        return value

    def parse_factor(self):
        token = self.peek()
        if token == "-":
            self.pop("-")
            return -self.parse_factor()
        if token == "(":
            self.pop("(")
            expr = self.parse_expr()
            self.pop(")")
            return expr
        if token == "NO":
            self.pop("NO")
            self.pop("[")
            args = [self.parse_expr()]
            while self.peek() == ",":
                self.pop(",")
                args.append(self.parse_expr())
            self.pop("]")
            return NO(*args)
        if token == "Derivative":
            self.pop("Derivative")
            self.pop("[")
            order = int(self.pop())
            self.pop("]")
            self.pop("[")
            base = self.parse_expr()
            self.pop("]")
            return d(base, order)
        self.pop()
        if token == "One":
            return One
        if re.fullmatch(r"\d+/\d+", token):
            p, q = token.split("/")
            return sp.Rational(int(p), int(q))
        if re.fullmatch(r"\d+", token):
            return sp.Integer(token)
        if token not in self.symbols:
            raise KeyError(f"Unknown symbol {token!r}")
        return self.symbols[token]


def parse_wl_expr(text, symbols):
    return _WLParser(text, symbols).parse()


def make_z3_free_field_data(prefix="wz3ff"):
    clear_registry()
    b = BasicOperator(f"b_{prefix}", fermionic=True,
                      conformal_weight=Fraction(2))
    c = BasicOperator(f"c_{prefix}", fermionic=True,
                      conformal_weight=Fraction(-1))
    beta = BasicOperator(f"beta_{prefix}", conformal_weight=Fraction(3, 2))
    gamma = BasicOperator(f"gamma_{prefix}", conformal_weight=Fraction(-1, 2))

    Bosonic(beta, gamma)
    Fermionic(b, c)
    OPE[b, c] = MakeOPE([One])
    OPE[beta, gamma] = MakeOPE([-One])

    J = 2 * NO(b, c) + 3 * NO(beta, gamma)
    G = NO(gamma, b)
    Gbar = 2 * NO(d(beta), c) + 3 * NO(beta, d(c))
    T = (
        -2 * NO(b, d(c))
        - Fraction(3, 2) * NO(beta, d(gamma))
        - NO(d(b), c)
        - Fraction(1, 2) * NO(d(beta), gamma)
    )
    W = beta
    Wbar = (
        NO(beta, NO(beta, NO(gamma, NO(gamma, gamma))))
        + 2 * NO(beta, NO(gamma, NO(gamma, NO(b, c))))
        - 4 * NO(beta, NO(d(gamma), gamma))
        - Fraction(4, 3) * NO(gamma, NO(b, d(c)))
        + Fraction(2, 3) * NO(gamma, NO(d(b), c))
        + Fraction(2, 3) * NO(d(beta), NO(gamma, gamma))
        - Fraction(8, 3) * NO(d(gamma), NO(b, c))
        + Fraction(10, 9) * d(d(gamma))
    )
    GW = bracket(G, W, 1)
    GbarWbar = bracket(Gbar, Wbar, 1)
    return {
        "b": b,
        "c": c,
        "beta": beta,
        "gamma": gamma,
        "T": T,
        "J": J,
        "W": W,
        "Wbar": Wbar,
        "G": G,
        "Gbar": Gbar,
        "GW": GW,
        "GbarWbar": GbarWbar,
    }


def _pair_to_expr(left, right, pole_map, ops):
    max_pole = max((int(k) for k in pole_map), default=0)
    data = []
    for pole in range(max_pole, 0, -1):
        expr = parse_wl_expr(pole_map.get(str(pole), "0"), ops)
        data.append(expr)
    OPE[left, right] = MakeOPE(data)


def make_z3_abstract_data(prefix="wz3abs"):
    clear_registry()
    T = BasicOperator(f"T_{prefix}", conformal_weight=2)
    J = BasicOperator(f"J_{prefix}", conformal_weight=1)
    W = BasicOperator(f"W_{prefix}", conformal_weight=Fraction(3, 2))
    Wbar = BasicOperator(f"Wbar_{prefix}", conformal_weight=Fraction(3, 2))
    G = BasicOperator(f"G_{prefix}", fermionic=True,
                      conformal_weight=Fraction(3, 2))
    Gbar = BasicOperator(
        f"Gbar_{prefix}", fermionic=True, conformal_weight=Fraction(3, 2)
    )
    GW = BasicOperator(f"GW_{prefix}", fermionic=True, conformal_weight=2)
    GbarWbar = BasicOperator(
        f"GbarWbar_{prefix}", fermionic=True, conformal_weight=2)
    Bosonic(T, J, W, Wbar)
    Fermionic(G, Gbar, GW, GbarWbar)
    ops = {
        "T": T,
        "J": J,
        "W": W,
        "Wbar": Wbar,
        "G": G,
        "Gbar": Gbar,
        "GW": GW,
        "GbarWbar": GbarWbar,
        "One": One,
    }
    for key, pole_map in W_Z3_OPE_GROUND_TRUTH.items():
        left_name, right_name = key.split("_", 1)
        _pair_to_expr(ops[left_name], ops[right_name], pole_map, ops)
    return ops


def expected_ope_expr_map(ops):
    parsed = {}
    symbols = {**ops, "One": One}
    for key, pole_map in W_Z3_OPE_GROUND_TRUTH.items():
        parsed[key] = {
            int(pole): simplify(parse_wl_expr(expr, symbols))
            for pole, expr in pole_map.items()
        }
    return parsed


def load_selected_null_relation_sources():
    text = W_Z3_MD.read_text(encoding="utf-8")
    results = {}
    for basis_id in W_Z3_NULL_GROUND_TRUTH["basis_ids"]:
        pattern = rf"## Basis {basis_id}.*?```wl\n(.*?)\n```"
        match = re.search(pattern, text, re.S)
        if not match:
            raise ValueError(f"Missing Basis {basis_id} block")
        body = match.group(1).strip()
        body = body.replace("== 0", "").strip()
        if body.startswith("(") and body.endswith(")"):
            body = body[1:-1].strip()
        results[f"Basis {basis_id}"] = body
    special = re.search(
        r"## Particular T4 Relation.*?```wl\n(.*?)\n```", text, re.S)
    if not special:
        raise ValueError("Missing Particular T4 Relation block")
    body = special.group(1).strip().replace("== 0", "").strip()
    if body.startswith("(") and body.endswith(")"):
        body = body[1:-1].strip()
    results["Particular T4 Relation"] = body
    return results


def build_null_relations(ops):
    symbols = {**ops, "One": One}
    return {
        name: simplify(parse_wl_expr(source, symbols))
        for name, source in load_selected_null_relation_sources().items()
    }


def assert_zero_jacobi_matrix(matrix):
    for row in matrix:
        for value in row:
            assert value == Zero


def compute_python_jacobi_summary():
    ops = make_z3_abstract_data()
    summary = {}
    for triple in W_Z3_JACOBI_GROUND_TRUTH["triples"]:
        matrix = check_jacobi_identity(*(ops[name] for name in triple))
        assert_zero_jacobi_matrix(matrix)
        summary["|".join(triple)] = {
            "rows": len(matrix),
            "cols": len(matrix[0]) if matrix else 0,
            "all_zero": True,
        }
    return summary


def wolframscript_available():
    return shutil.which("wolframscript") is not None


def run_wolfram_jacobi_summary():
    triples = W_Z3_JACOBI_GROUND_TRUTH["triples"]
    triple_code = ", ".join(
        "{" + ", ".join(f'"{name}"' for name in triple) + "}" for triple in triples
    )
    script = f'''
Get["{(ROOT / "OPEdefs" / "OPEdefs.m").as_posix()}"];
Bosonic[T, J, W, Wbar];
Fermionic[G, Gbar, GW, GbarWbar];
OPE[T, T] = MakeOPE[{{-15/2 One, 0, 2 T, Derivative[1][T]}}];
OPE[T, J] = MakeOPE[{{0, J, Derivative[1][J]}}];
OPE[J, J] = MakeOPE[{{-5 One, 0}}];
OPE[T, G] = MakeOPE[{{0, 3/2 G, Derivative[1][G]}}];
OPE[T, Gbar] = MakeOPE[{{0, 3/2 Gbar, Derivative[1][Gbar]}}];
OPE[T, W] = MakeOPE[{{0, 3/2 W, Derivative[1][W]}}];
OPE[T, Wbar] = MakeOPE[{{0, 3/2 Wbar, Derivative[1][Wbar]}}];
OPE[T, GW] = MakeOPE[{{0, 2 GW, Derivative[1][GW]}}];
OPE[T, GbarWbar] = MakeOPE[{{0, 2 GbarWbar, Derivative[1][GbarWbar]}}];
OPE[J, G] = MakeOPE[{{-G}}];
OPE[J, Gbar] = MakeOPE[{{Gbar}}];
OPE[J, W] = MakeOPE[{{3 W}}];
OPE[J, Wbar] = MakeOPE[{{-3 Wbar}}];
OPE[J, GW] = MakeOPE[{{2 GW}}];
OPE[J, GbarWbar] = MakeOPE[{{-2 GbarWbar}}];
OPE[W, Wbar] = MakeOPE[{{-20/9 One, 4/3 J, 2/3 T - 1/3 NO[J, J] + 2/3 Derivative[1][J]}}];
OPE[G, W] = MakeOPE[{{GW}}];
OPE[Gbar, W] = MakeOPE[{{0}}];
OPE[Gbar, Wbar] = MakeOPE[{{GbarWbar}}];
OPE[G, Wbar] = MakeOPE[{{0}}];
OPE[G, Gbar] = MakeOPE[{{5 One, J, -T + 1/2 Derivative[1][J]}}];
OPE[Gbar, GW] = MakeOPE[{{-3 W, -Derivative[1][W]}}];
OPE[G, GbarWbar] = MakeOPE[{{-3 Wbar, -Derivative[1][Wbar]}}];
OPE[Wbar, GW] = MakeOPE[{{4/3 G, 2/3 NO[J, G] + 2/3 Derivative[1][G]}}];
OPE[W, GbarWbar] = MakeOPE[{{-4/3 Gbar, 2/3 NO[J, Gbar] - 2/3 Derivative[1][Gbar]}}];
OPE[GW, GbarWbar] = MakeOPE[{{-20/3 One, 8/3 J, 2 T - 1/3 NO[J, J] + 4/3 Derivative[1][J], 2/3 NO[G, Gbar] - 2/3 NO[J, T] - 1/3 NO[J, Derivative[1][J]] + 4/3 Derivative[1][T] + 1/3 Derivative[2][J]}}];
triples = {{{triple_code}}};
summary = Association@Table[
  Module[{{res, key}},
    key = StringRiffle[triple, "|"];
    res = OPEJacobi @@ (ToExpression /@ triple);
    key -> <|"rows" -> Length[res], "cols" -> If[Length[res] == 0, 0, Length[res[[1]]]], "all_zero" -> And @@ Flatten[Map[# === 0 &, res, {{2}}]]|>
  ],
  {{triple, triples}}
];
Print[ExportString[Normal[summary], "JSON"]];
'''
    with tempfile.TemporaryDirectory(prefix="wz3-jacobi-") as tmpdir:
        path = Path(tmpdir) / "jacobi.wls"
        path.write_text(script, encoding="utf-8")
        result = subprocess.run(
            ["wolframscript", "-file", str(path)],
            capture_output=True,
            text=True,
            encoding="utf-8",
            check=False,
        )
    if result.returncode != 0:
        raise RuntimeError(result.stderr or result.stdout)
    payload_lines = []
    recording = False
    for line in result.stdout.splitlines():
        stripped = line.strip()
        if not recording and (stripped.startswith("[") or stripped.startswith("{")):
            recording = True
        if recording:
            payload_lines.append(line)
    if not payload_lines:
        raise RuntimeError(f"No JSON payload found in output: {result.stdout}")
    return json.loads("\n".join(payload_lines))

In [5]:

def _pair_to_expr(left, right, pole_map, ops):
    max_pole = max((int(k) for k in pole_map), default=0)
    data = []
    for pole in range(max_pole, 0, -1):
        expr = parse_wl_expr(pole_map.get(str(pole), "0"), ops)
        data.append(expr)
    OPE[left, right] = MakeOPE(data)

In [9]:
clear_registry()
T = BasicOperator(f"T", conformal_weight=2)
J = BasicOperator(f"J", conformal_weight=1)
W = BasicOperator(f"W", conformal_weight=Fraction(3, 2))
Wbar = BasicOperator(f"Wbar", conformal_weight=Fraction(3, 2))
G = BasicOperator(f"G", fermionic=True, conformal_weight=Fraction(3, 2))
Gbar = BasicOperator(
    f"Gbar", fermionic=True, conformal_weight=Fraction(3, 2)
)
GW = BasicOperator(f"GW", fermionic=True, conformal_weight=2)
GbarWbar = BasicOperator(f"GbarWbar", fermionic=True, conformal_weight=2)
Bosonic(T, J, W, Wbar)
Fermionic(G, Gbar, GW, GbarWbar)
ops = {
    "T": T,
    "J": J,
    "W": W,
    "Wbar": Wbar,
    "G": G,
    "Gbar": Gbar,
    "GW": GW,
    "GbarWbar": GbarWbar,
    "One": One,
}
for key, pole_map in W_Z3_OPE_GROUND_TRUTH.items():
    left_name, right_name = key.split("_", 1)
    _pair_to_expr(ops[left_name], ops[right_name], pole_map, ops)



In [10]:
OPE(GbarWbar, G)


OPEData({2: 3*Wbar, 1: 2*∂Wbar})

In [15]:
def flatten_and_deduplicate(lst):
    def flatten_completely(lst):
        result = []
        for item in lst:
            if isinstance(item, list):
                result.extend(flatten_completely(item))
            else:
                result.append(item)
        return result
    flatten = flatten_completely(lst)

    unique = []
    for item in flatten:
        if item not in unique:
            unique.append(item)
    return unique
  

generators = [T, W, J, G, Gbar, GW, Wbar, GbarWbar]
jacobi_identities = [[check_jacobi_identity(GbarWbar, g2, g3)
                     for g2 in generators] for g3 in generators]


non_zero = flatten_and_deduplicate(jacobi_identities)
[simplify(expr) for expr in non_zero]

[ConstantOperator('Zero'),
 -2*∂^2W/3 - 2*NO(Gbar,GW)/3 - 2*NO(J,∂W)/3 + 2*NO(T,W) + NO(∂J,W),
 4*∂GbarWbar/3 + 2*NO(J,GbarWbar)/3 + 2*NO(Wbar,Gbar),
 8*NO(Gbar,GbarWbar)/3,
 2*∂^2W/3 + 2*NO(Gbar,GW)/3 + 2*NO(J,∂W)/3 - 2*NO(T,W) - NO(∂J,W),
 -2*NO(J,∂GW)/3 + 8*NO(T,GW)/3 + 2*NO(W,∂G) + 2*NO(∂J,GW)/3 - 2*NO(∂W,G)/3,
 4*∂GW/3 - 2*NO(J,GW)/3 + 2*NO(W,G),
 -2*∂^2Wbar/3 - 2*NO(G,GbarWbar)/3 + 2*NO(J,∂Wbar)/3 + 2*NO(T,Wbar) - NO(∂J,Wbar),
 4*∂^2GbarWbar/3 - 8*NO(T,GbarWbar)/3 + 4*NO(∂J,GbarWbar)/3 + 8*NO(∂Wbar,Gbar)/3,
 -4*∂GbarWbar/3 - 2*NO(J,GbarWbar)/3 - 2*NO(Wbar,Gbar),
 2*∂^2Wbar/3 + 2*NO(G,GbarWbar)/3 - 2*NO(J,∂Wbar)/3 - 2*NO(T,Wbar) + NO(∂J,Wbar),
 -8*NO(Gbar,GbarWbar)/3,
 -2*NO(J,∂GbarWbar)/3 - 8*NO(T,GbarWbar)/3 - 2*NO(Wbar,∂Gbar) + 2*NO(∂J,GbarWbar)/3 + 2*NO(∂Wbar,Gbar)/3]

In [22]:
non_zero[2]

4*∂GbarWbar/3 + 2*NO(J,GbarWbar)/3 + 2*NO(Wbar,Gbar)